In [1]:
import requests
from bs4 import BeautifulSoup
import inspect
import json
import os   
from pathlib import Path
import pandas as pd
import re
import io
import librosa
import numpy as np
import matplotlib.pyplot as plt
import librosa.display
import soundfile as sf


In [2]:
tamanhoDataset = 400

<h3>Neste trecho, uma url é analisada, retornando uma série de códigos de aves </h3>    

<p>Ainda não está finalizado. A ideia final é criar um algorítmo que possa iterar pela página, mapeando as espécies e jogando elas dentro da minha máquina</p>

In [3]:
def selectAves(FamiliaSelect):

    urlInfo="https://www.wikiaves.com.br/especies.php?t=t"

    # Está parte acessa a lista de espécies do wiki aves direto da página

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/121.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Referer": "https://www.wikiaves.com.br/"
    }

    responseInfo = requests.get(urlInfo, headers=headers)
    soup = BeautifulSoup(responseInfo.content, "html.parser")

    soup = soup.find_all("table", class_="wa-table-sp table m-table m-table--head-separator-metal wa-table-hover")[0]
    aves = soup.find("tbody").find_all("script")[1:]

    # A lista das espécies está misturada com código html
    # Este trecho serve para limpar esta lista e transforma-la
    # em um dataframe pandas organizado

    avesDf = []

    # iteramos por item dentro do html
    for script in aves:
        texto = script.get_text(strip=True)
        
        # removemos a função "lsp" do html
        match = re.search(r"lsp\((.*)\);", texto)

        valores = []

        # fazemos separação dos valores por ,
        for v in match.group(1).split(","):
            v.strip().strip("'") 
            valores.append(v)

        # A listagem das espécies no site não lista a família da ave em todos os registros
        # apenas no primeiro registro que identifica aquela família, por isso o replicamos
        # para que no meu dataframe tenhamos a família da ave colocada corretamente nas colunas
        if valores[1] != " ''":
            familiaAtual = valores[1]

        else:
            valores[1] = familiaAtual
        
        avesDf.append({
            "codigo": int(valores[0].strip().strip("'")),
            "familia": valores[1].strip().strip("'") or None,
            "nome_cientifico": valores[2].strip().strip("'"),
            "nome_popular": valores[3].strip().strip("'"),
            "slug": valores[4].strip().strip("'"),
            "fotos": int(valores[5]),
            "sons": int(valores[6]),
        })

    avesDf = pd.DataFrame(avesDf)

    avesSelect = avesDf[
        (avesDf["familia"] == FamiliaSelect) &
        (avesDf["sons"] > tamanhoDataset)
    ][["codigo", "nome_popular"]]

    return avesSelect

<h3>função de pré processamento, para facilitar a leitura do código</h3>

In [4]:
def preProcess(audio, sr):
    TARGET_SECONDS = 5
    TARGET_SAMPLES = TARGET_SECONDS * sr

    FRAME_LEN = 2048
    HOP = 512

    # Calcula a média quadrática para detectar evento principal 
    # Esse código desloca uma janela de amostragem pelo áudio 
    # calculando a média de energia, assim podemos saber qual é
    # o trecho com mais atividade

    rms = librosa.feature.rms(
        y=audio,
        frame_length=FRAME_LEN,
        hop_length=HOP
    )[0]

    # Pegamos o trecho com mais atividade do retorno da função 
    # e então aplicamos essa multiplicação para removermos os 20% 
    # menos relevante do audio, assim eliminando uma faixa de ruído
    
    threshold = 0.2 * np.max(rms)
    frames_event = np.where(rms > threshold)[0]

    # Se não detectar evento (silêncio, erro, etc)
    if len(frames_event) == 0:
        return librosa.util.fix_length(audio, size=TARGET_SAMPLES)

    start_sample = librosa.frames_to_samples(frames_event[0], hop_length=HOP)
    end_sample   = librosa.frames_to_samples(frames_event[-1], hop_length=HOP)

    audio_event = audio[start_sample:end_sample]

    # Centraliza o evento na janela fixa
    if len(audio_event) >= TARGET_SAMPLES:
        audio_event = audio_event[:TARGET_SAMPLES]
    else:
        pad_total = TARGET_SAMPLES - len(audio_event)
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left

        audio_event = np.pad(audio_event, (pad_left, pad_right))

    return audio_event


<h3>Aqui pegamos um código de ave diretamente, e então o transformamos em uma pasta com todos os seus sons disponíveis.</h3>

<h3>Aqui pegamos um código de ave diretamente, e então o transformamos em uma pasta com todos os seus sons disponíveis em formato npy</h3>

In [5]:
def createDataset(codigoAve, nomeAve, dwldLimit):
    import requests, json, io
    import numpy as np
    import librosa
    from pathlib import Path

    TARGET_SR = 44100  # sample rate fixo

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/121.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Referer": "https://www.wikiaves.com.br/"
    }

    dataset_path = Path("..") / "dataset" / nomeAve
    dataset_path.mkdir(parents=True, exist_ok=True)

    cont = 0
    page = 1

    while cont < dwldLimit:

        urlFiles = (
            "https://www.wikiaves.com.br/getRegistrosJSON.php?"
            f"tm=s&t=s&s={codigoAve}&p={page}"
        )

        responseFiles = requests.get(urlFiles, headers=headers)
        data = json.loads(responseFiles.text)["registros"]["itens"]

        if not data:
            print(f"Sem mais registros na página {page}")
            break

        for registro in data:
            link = data[registro]["link"]

            link = link.replace(".jpg", ".mp3")
            link = link.replace("#", "")

            try:
                responseAudio = requests.get(link, timeout=10)

                audio, sr = librosa.load(
                    io.BytesIO(responseAudio.content),
                    sr=None,
                    mono=True
                )

                # Reamostragem
                if sr != TARGET_SR:
                    audio = librosa.resample(
                        audio,
                        orig_sr=sr,
                        target_sr=TARGET_SR
                    )
                    sr = TARGET_SR

                # Pré-processamento
                audio = preProcess(audio, sr)

            except Exception as e:
                print(f"Erro no arquivo {registro}: {e}")
                continue

            # ===== MEL SPECTROGRAM =====
            mel_spec = librosa.feature.melspectrogram(
                y=audio,
                sr=sr,
                n_fft=2048,
                hop_length=512,
                n_mels=128,
                fmin=0,
                fmax=sr // 2
            )

            # Converter para dB
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

            path = dataset_path / f"{nomeAve}_{cont}.npy"
            np.save(path, mel_spec_db)

            cont += 1

            if cont >= dwldLimit:
                break

        page += 1

Vamos usar a função que criamos, e iterar por avesSelect, assim fazedndo os downloads

In [7]:
avesSelect = selectAves("Threskiornithidae")

for x in range(0, len(avesSelect)):
    #createDatasetAudio(avesSelect.iloc[x]["codigo"], avesSelect.iloc[x]["nome_popular"], tamanhoDataset)
    createDataset(avesSelect.iloc[x]["codigo"], avesSelect.iloc[x]["nome_popular"], tamanhoDataset)

Erro no arquivo 7: Error opening <_io.BytesIO object at 0x000002043F813650>: Format not recognised.
Erro no arquivo 6: Error opening <_io.BytesIO object at 0x000002043FDC31F0>: Format not recognised.
Erro no arquivo 8: Error opening <_io.BytesIO object at 0x000002043F813DD0>: Format not recognised.
Erro no arquivo 16: Error opening <_io.BytesIO object at 0x0000020443081CB0>: Format not recognised.
Erro no arquivo 13: Error opening <_io.BytesIO object at 0x000002043F5629D0>: Format not recognised.
Erro no arquivo 1: HTTPSConnectionPool(host='s3.amazonaws.com', port=443): Read timed out. (read timeout=10)
Erro no arquivo 3: Error opening <_io.BytesIO object at 0x000002043FDC3650>: Format not recognised.
Erro no arquivo 10: Error opening <_io.BytesIO object at 0x000002043FDA0F90>: Format not recognised.
Erro no arquivo 18: Error opening <_io.BytesIO object at 0x000002043F8D7970>: Format not recognised.
Erro no arquivo 5: Error opening <_io.BytesIO object at 0x0000020424A8E750>: Format not

In [8]:
avesSelect = selectAves("Strigidae")

for x in range(0, len(avesSelect)):
    #createDatasetAudio(avesSelect.iloc[x]["codigo"], avesSelect.iloc[x]["nome_popular"], tamanhoDataset)
    createDataset(avesSelect.iloc[x]["codigo"], avesSelect.iloc[x]["nome_popular"], tamanhoDataset)

Erro no arquivo 16: Error opening <_io.BytesIO object at 0x000002043FBEA160>: Format not recognised.
Erro no arquivo 12: Error opening <_io.BytesIO object at 0x000002043FDA0FE0>: Format not recognised.
Erro no arquivo 19: Error opening <_io.BytesIO object at 0x000002044396C900>: Format not recognised.
Erro no arquivo 6: Error opening <_io.BytesIO object at 0x000002043F590D60>: Format not recognised.
Erro no arquivo 7: Error opening <_io.BytesIO object at 0x000002043FBEA200>: Format not recognised.
Erro no arquivo 14: Error opening <_io.BytesIO object at 0x000002043F590D60>: Format not recognised.
Erro no arquivo 17: Error opening <_io.BytesIO object at 0x000002043F658EA0>: Format not recognised.
Erro no arquivo 7: Error opening <_io.BytesIO object at 0x0000020424A8EB10>: Format not recognised.
Erro no arquivo 17: Error opening <_io.BytesIO object at 0x000002043F4D94E0>: Format not recognised.
Erro no arquivo 20: Error opening <_io.BytesIO object at 0x000002043FDC3EC0>: Format not recog

In [ ]:
avesSelect

,codigo,nome_popular
2,10003,macuco
5,10006,inhambu-pixuna
6,10007,tururim
7,10008,inhambuguaçu
8,10009,jaó
13,10013,jaó-do-sul
18,10018,inhambu-chororó
19,10019,inhambu-chintã
20,10020,perdiz
23,10023,codorna-amarela


In [ ]:
def CreateEspectograms(especie): 

    # pasta onde estão os .npy
    DATASET_DIR = Path("../dataset/"+especie)

    # lista todos os arquivos .npy
    files = sorted(DATASET_DIR.glob("*.npy"))

    for file in files:
        spec = np.load(file)

        plt.figure(figsize=(10, 4))
        librosa.display.specshow(
            spec,
            #sr=sr,
            hop_length=512,
            x_axis="time",
            y_axis="hz"
        )
        plt.colorbar(format="%+2.0f dB")
        plt.title("Espectrograma (STFT)")
        plt.tight_layout()
        #plt.show()

        output_path = file.with_suffix(".jpg")
        plt.savefig(output_path, dpi=300)
        plt.close()